# 05 - LangChain com memória externa (SQLite)

No notebook anterior o histórico ficava em memória (`InMemoryChatMessageHistory`) e sumia quando o processo terminava.

Aqui guardamos o histórico em um banco **SQLite**, usando só a biblioteca padrão `sqlite3`. Para isso criamos uma classe que herda de `BaseChatMessageHistory`, a interface que o `RunnableWithMessageHistory` espera.

As configurações de chave e modelo continuam vindo do `.env` na raiz do projeto.

In [1]:
import os
import sqlite3
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

CAMINHO_BANCO = Path("historico_chat.db")

## 1. Classe de histórico em SQLite

A classe implementa três métodos de `BaseChatMessageHistory`:

- `messages`: lê as mensagens da sessão no banco.
- `add_messages`: grava novas mensagens.
- `clear`: apaga o histórico da sessão.

As mensagens são serializadas em JSON com `messages_to_dict` e `messages_from_dict`.

In [2]:
import json
from typing import Sequence

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, messages_from_dict, messages_to_dict


class HistoricoSQLite(BaseChatMessageHistory):
    def __init__(self, session_id: str, caminho: Path = CAMINHO_BANCO):
        self.session_id = session_id
        self.caminho = caminho
        with sqlite3.connect(self.caminho) as conexao:
            conexao.execute(
                """
                CREATE TABLE IF NOT EXISTS mensagens (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    conteudo TEXT NOT NULL
                )
                """
            )

    @property
    def messages(self) -> list[BaseMessage]:
        with sqlite3.connect(self.caminho) as conexao:
            linhas = conexao.execute(
                "SELECT conteudo FROM mensagens WHERE session_id = ? ORDER BY id",
                (self.session_id,),
            ).fetchall()
        return messages_from_dict([json.loads(conteudo) for (conteudo,) in linhas])

    def add_messages(self, messages: Sequence[BaseMessage]) -> None:
        with sqlite3.connect(self.caminho) as conexao:
            conexao.executemany(
                "INSERT INTO mensagens (session_id, conteudo) VALUES (?, ?)",
                [
                    (self.session_id, json.dumps(dados, ensure_ascii=False))
                    for dados in messages_to_dict(list(messages))
                ],
            )

    def clear(self) -> None:
        with sqlite3.connect(self.caminho) as conexao:
            conexao.execute("DELETE FROM mensagens WHERE session_id = ?", (self.session_id,))

## 2. Chain com histórico no SQLite

A diferença para o notebook 04 está em `get_session_history`: em vez de devolver um objeto em memória, ele abre o histórico da sessão no banco.

In [3]:
from langchain_core.runnables.history import RunnableWithMessageHistory


def montar_cadeia():
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "Você é um assistente prestativo. Responda em português, em uma ou duas frases."),
            MessagesPlaceholder("historico"),
            ("human", "{mensagem}"),
        ]
    )
    return RunnableWithMessageHistory(
        prompt | llm | StrOutputParser(),
        lambda session_id: HistoricoSQLite(session_id),
        input_messages_key="mensagem",
        history_messages_key="historico",
    )


cadeia = montar_cadeia()
config = {"configurable": {"session_id": "aula-1"}}

for pergunta in [
    "Oi, meu nome é Leonardo e estou estudando LangChain.",
    "Qual é o meu nome e o que eu estou estudando?",
]:
    resposta = cadeia.invoke({"mensagem": pergunta}, config=config)
    display(Markdown(f"**Você:** {pergunta}\n\n**Modelo:** {resposta}"))

/tmp/ipykernel_230933/3201806703.py:20: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  cadeia = montar_cadeia()


**Você:** Oi, meu nome é Leonardo e estou estudando LangChain.

**Modelo:** Oi Leonardo — prazer! Posso ajudar com conceitos, exemplos práticos em Python, templates de prompt, conectores e deployment; o que você quer aprender primeiro?

**Você:** Qual é o meu nome e o que eu estou estudando?

**Modelo:** Seu nome é Leonardo e você está estudando LangChain.

## 3. Persistência entre execuções

Vamos apagar a chain e o objeto de histórico da memória do Python. O que restou foi o arquivo `historico_chat.db`. Montando a chain de novo, a sessão `aula-1` continua lembrando.

In [4]:
del cadeia

cadeia = montar_cadeia()
resposta = cadeia.invoke({"mensagem": "Qual é o meu nome?"}, config=config)

display(Markdown(f"**Você (após recriar a chain):** Qual é o meu nome?\n\n**Modelo:** {resposta}"))

/tmp/ipykernel_230933/3139022480.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  cadeia = montar_cadeia()


**Você (após recriar a chain):** Qual é o meu nome?

**Modelo:** Seu nome é Leonardo.

Conferindo as linhas gravadas diretamente no banco:

In [5]:
with sqlite3.connect(CAMINHO_BANCO) as conexao:
    linhas = conexao.execute(
        "SELECT id, session_id, json_extract(conteudo, '$.type'), json_extract(conteudo, '$.data.content') FROM mensagens ORDER BY id"
    ).fetchall()

for linha in linhas:
    print(linha)

(1, 'aula-1', 'human', 'Oi, meu nome é Leonardo e estou estudando LangChain.')
(2, 'aula-1', 'ai', 'Oi Leonardo — prazer! Posso ajudar com conceitos, exemplos práticos em Python, templates de prompt, conectores e deployment; o que você quer aprender primeiro?')
(3, 'aula-1', 'human', 'Qual é o meu nome e o que eu estou estudando?')
(4, 'aula-1', 'ai', 'Seu nome é Leonardo e você está estudando LangChain.')
(5, 'aula-1', 'human', 'Qual é o meu nome?')
(6, 'aula-1', 'ai', 'Seu nome é Leonardo.')


## 4. Sessões separadas e limpeza

Cada `session_id` tem o próprio histórico no mesmo banco. Uma sessão nova não enxerga a `aula-1`.

In [6]:
resposta = cadeia.invoke(
    {"mensagem": "Qual é o meu nome?"},
    config={"configurable": {"session_id": "aula-2"}},
)

display(Markdown(f"**Você (sessão aula-2):** Qual é o meu nome?\n\n**Modelo:** {resposta}"))

**Você (sessão aula-2):** Qual é o meu nome?

**Modelo:** Não sei seu nome porque você ainda não me disse. Quer me dizer como devo chamá-lo(a)?

In [7]:
HistoricoSQLite("aula-1").clear()
print("Mensagens da aula-1 após clear:", len(HistoricoSQLite("aula-1").messages))

Mensagens da aula-1 após clear: 0


## Observações

- A classe é um exemplo didático. Em produção, convém tratar concorrência, criar um índice por `session_id` e reaproveitar a conexão em vez de abri-la a cada operação.
- O mesmo padrão serve para qualquer banco: basta implementar `messages`, `add_messages` e `clear` com a biblioteca que você usar.
- O LangChain também tem `SQLChatMessageHistory` pronta no pacote `langchain-community`. Aqui escrevemos a própria classe para mostrar a interface.